In [ ]:
"""
DataFrame Validation System using Strategy + Chain of Responsibility Patterns

Based on:
- Gang of Four (1994). Design Patterns: Elements of Reusable Object-Oriented Software
  - Chain of Responsibility Pattern (pg. 223-232)
  - Strategy Pattern (pg. 315-323)
- Martin Fowler (2018). Refactoring: Improving the Design of Existing Code

Design Benefits:
1. Hierarchical validation structure mirrors your spec
2. Easy to add/remove validators without breaking existing code
3. Each validator is independently testable
4. Clear separation of concerns
5. Validation results are traceable for debugging
"""

from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Callable, Protocol
from abc import ABC, abstractmethod
from enum import Enum
import pandas as pd
import numpy as np
from datetime import datetime
import uuid
from loguru import logger


# ============= VALIDATION RESULT =============
@dataclass
class ValidationResult:
    """Stores validation results with context for debugging."""
    validator_name: str
    validator_type: str  # e.g., "number.int", "string.format"
    passed: bool
    invalid_indices: pd.Index = field(default_factory=pd.Index)
    error_message: Optional[str] = None
    metadata: Dict[str, Any] = field(default_factory=dict)
    
    def __bool__(self) -> bool:
        return self.passed
    
    def __repr__(self) -> str:
        status = "✓ PASS" if self.passed else "✗ FAIL"
        count = len(self.invalid_indices)
        return f"{status} [{self.validator_type}] {self.validator_name}: {count} errors"


# ============= BASE VALIDATOR (Strategy Pattern) =============
class Validator(ABC):
    """
    Base validator using Strategy Pattern.
    Each validator encapsulates a specific validation algorithm.
    
    Reference: Gang of Four, pg. 315-323
    """
    
    def __init__(self, name: str, validator_type: str):
        self.name = name
        self.validator_type = validator_type
    
    @abstractmethod
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        """Execute validation and return result."""
        pass
    
    def _create_result(self, passed: bool, invalid_indices: pd.Index = None, 
                       error_message: str = None, **metadata) -> ValidationResult:
        """Helper to create validation result."""
        if invalid_indices is None:
            invalid_indices = pd.Index([])
        
        return ValidationResult(
            validator_name=self.name,
            validator_type=self.validator_type,
            passed=passed,
            invalid_indices=invalid_indices,
            error_message=error_message,
            metadata=metadata
        )


# ============= NUMBER VALIDATORS =============
class IsNumericValidator(Validator):
    """Validate if values are numeric."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~pd.to_numeric(sr, errors='coerce').notna()].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class IsIntValidator(Validator):
    """Validate if values are integers."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~sr.apply(lambda x: isinstance(x, (int, np.integer)))].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class IsFloatValidator(Validator):
    """Validate if values are floats."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~sr.apply(lambda x: isinstance(x, (float, np.floating)))].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class NumericRangeValidator(Validator):
    """Validate if numeric values are within range."""
    
    def validate(self, data: pd.Series, min_val: float = None, 
                 max_val: float = None, **kwargs) -> ValidationResult:
        sr = data.dropna()
        mask = pd.Series(True, index=sr.index)
        
        if min_val is not None:
            mask &= (sr >= min_val)
        if max_val is not None:
            mask &= (sr <= max_val)
        
        invalid = sr[~mask].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            min_value=min_val,
            max_value=max_val
        )


class DecimalFormatValidator(Validator):
    """Validate decimal format (e.g., 2 decimal places)."""
    
    def validate(self, data: pd.Series, decimal_places: int = 2, **kwargs) -> ValidationResult:
        sr = data.dropna()
        
        def check_decimals(x):
            if not isinstance(x, (float, np.floating)):
                return False
            str_val = f"{x:.10f}".rstrip('0')
            if '.' in str_val:
                actual_decimals = len(str_val.split('.')[1])
                return actual_decimals <= decimal_places
            return True
        
        invalid = sr[~sr.apply(check_decimals)].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            decimal_places=decimal_places
        )


# ============= STRING VALIDATORS =============
class IsStringValidator(Validator):
    """Validate if values are strings."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~sr.apply(lambda x: isinstance(x, str))].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class StringLengthValidator(Validator):
    """Validate string length range."""
    
    def validate(self, data: pd.Series, min_length: int = None, 
                 max_length: int = None, **kwargs) -> ValidationResult:
        sr = data.dropna().astype(str)
        mask = pd.Series(True, index=sr.index)
        
        if min_length is not None:
            mask &= (sr.str.len() >= min_length)
        if max_length is not None:
            mask &= (sr.str.len() <= max_length)
        
        invalid = sr[~mask].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            min_length=min_length,
            max_length=max_length
        )


class StringFormatValidator(Validator):
    """Validate string format using regex."""
    
    def validate(self, data: pd.Series, pattern: str, **kwargs) -> ValidationResult:
        sr = data.dropna().astype(str)
        invalid = sr[~sr.str.match(pattern)].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            pattern=pattern
        )


class ValueListValidator(Validator):
    """Validate if values are in allowed list."""
    
    def validate(self, data: pd.Series, allowed_values: List[Any], **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~sr.isin(allowed_values)].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            allowed_values=allowed_values
        )


class SubstringValidator(Validator):
    """Validate if substring exists in string."""
    
    def validate(self, data: pd.Series, substring: str, must_contain: bool = True, 
                 **kwargs) -> ValidationResult:
        sr = data.dropna().astype(str)
        
        if must_contain:
            invalid = sr[~sr.str.contains(substring, na=False)].index
        else:
            invalid = sr[sr.str.contains(substring, na=False)].index
        
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            substring=substring,
            must_contain=must_contain
        )


class UUIDValidator(Validator):
    """Validate if values are valid UUIDs."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna().astype(str)
        
        def is_valid_uuid(x):
            try:
                uuid.UUID(x)
                return True
            except (ValueError, AttributeError):
                return False
        
        invalid = sr[~sr.apply(is_valid_uuid)].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class ReferenceValidator(Validator):
    """Validate against reference values (inner/outer)."""
    
    def validate(self, data: pd.Series, reference_values: List[Any], 
                 check_type: str = "inner", **kwargs) -> ValidationResult:
        """
        Args:
            check_type: 'inner' (must be in reference) or 'outer' (must NOT be in reference)
        """
        sr = data.dropna()
        
        if check_type == "inner":
            invalid = sr[~sr.isin(reference_values)].index
        elif check_type == "outer":
            invalid = sr[sr.isin(reference_values)].index
        else:
            raise ValueError(f"Invalid check_type: {check_type}. Use 'inner' or 'outer'")
        
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            reference_values=reference_values,
            check_type=check_type
        )


# ============= BOOLEAN VALIDATORS =============
class IsBooleanValidator(Validator):
    """Validate if values are boolean."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~sr.apply(lambda x: isinstance(x, (bool, np.bool_)))].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class BooleanMappingValidator(Validator):
    """Validate boolean mapping (e.g., Yes/No, True/False, 1/0)."""
    
    def validate(self, data: pd.Series, true_values: List[Any] = None, 
                 false_values: List[Any] = None, **kwargs) -> ValidationResult:
        if true_values is None:
            true_values = [True, 'True', 'true', 'yes', 'Yes', 'Y', 'y', 1]
        if false_values is None:
            false_values = [False, 'False', 'false', 'no', 'No', 'N', 'n', 0]
        
        sr = data.dropna()
        valid_values = true_values + false_values
        invalid = sr[~sr.isin(valid_values)].index
        
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            true_values=true_values,
            false_values=false_values
        )


# ============= DATE/DATETIME VALIDATORS =============
class IsDateValidator(Validator):
    """Validate if values are dates."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        sr = data.dropna()
        invalid = sr[~pd.to_datetime(sr, errors='coerce').notna()].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid
        )


class DateFormatValidator(Validator):
    """Validate date format."""
    
    def validate(self, data: pd.Series, date_format: str = "%Y-%m-%d", **kwargs) -> ValidationResult:
        sr = data.dropna()
        
        def check_format(x):
            try:
                if isinstance(x, (datetime, pd.Timestamp)):
                    return True
                datetime.strptime(str(x), date_format)
                return True
            except (ValueError, TypeError):
                return False
        
        invalid = sr[~sr.apply(check_format)].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            date_format=date_format
        )


class DateRangeValidator(Validator):
    """Validate if dates are within range."""
    
    def validate(self, data: pd.Series, min_date: str = None, 
                 max_date: str = None, **kwargs) -> ValidationResult:
        sr = pd.to_datetime(data.dropna(), errors='coerce')
        mask = pd.Series(True, index=sr.index)
        
        if min_date:
            mask &= (sr >= pd.to_datetime(min_date))
        if max_date:
            mask &= (sr <= pd.to_datetime(max_date))
        
        invalid = sr[~mask].index
        return self._create_result(
            passed=len(invalid) == 0,
            invalid_indices=invalid,
            min_date=min_date,
            max_date=max_date
        )


# ============= CATEGORY VALIDATORS =============
class IsCategoryValidator(Validator):
    """Validate if data is categorical type."""
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        is_category = isinstance(data.dtype, pd.CategoricalDtype)
        return self._create_result(
            passed=is_category,
            error_message=None if is_category else "Data is not categorical type"
        )


# ============= CUSTOM VALIDATOR =============
class CustomValidator(Validator):
    """Validator that accepts custom validation function."""
    
    def __init__(self, name: str, validator_type: str, validation_func: Callable):
        super().__init__(name, validator_type)
        self.validation_func = validation_func
    
    def validate(self, data: pd.Series, **kwargs) -> ValidationResult:
        """
        Custom function should return pd.Index of invalid values.
        """
        try:
            invalid = self.validation_func(data, **kwargs)
            return self._create_result(
                passed=len(invalid) == 0,
                invalid_indices=invalid
            )
        except Exception as e:
            logger.error(f"Custom validation '{self.name}' failed: {e}")
            return self._create_result(
                passed=False,
                error_message=str(e)
            )


# ============= VALIDATION CHAIN (Chain of Responsibility) =============
class ValidationChain:
    """
    Chain of Responsibility Pattern for sequential validation.
    
    Allows validators to be chained together, with each validator
    processing the data and optionally passing to the next.
    
    Reference: Gang of Four, pg. 223-232
    """
    
    def __init__(self, name: str = "ValidationChain"):
        self.name = name
        self.validators: List[Validator] = []
        self.stop_on_first_failure: bool = False
    
    def add_validator(self, validator: Validator) -> 'ValidationChain':
        """Add validator to chain (fluent interface)."""
        self.validators.append(validator)
        return self
    
    def set_stop_on_failure(self, stop: bool = True) -> 'ValidationChain':
        """Configure whether to stop on first failure."""
        self.stop_on_first_failure = stop
        return self
    
    def validate(self, data: pd.Series, **kwargs) -> List[ValidationResult]:
        """Execute all validators in chain."""
        results = []
        
        for validator in self.validators:
            result = validator.validate(data, **kwargs)
            results.append(result)
            
            if self.stop_on_first_failure and not result.passed:
                logger.info(f"Chain '{self.name}' stopped at validator '{validator.name}'")
                break
        
        return results
    
    def validate_and_summarize(self, data: pd.Series, **kwargs) -> Dict[str, Any]:
        """Execute validation and return summary."""
        results = self.validate(data, **kwargs)
        
        return {
            'chain_name': self.name,
            'total_validators': len(self.validators),
            'executed_validators': len(results),
            'all_passed': all(r.passed for r in results),
            'results': results,
            'failed_validators': [r for r in results if not r.passed]
        }


# ============= VALIDATION BUILDER (Builder Pattern) =============
class ValidationBuilder:
    """
    Builder pattern for constructing complex validation chains.
    Provides fluent interface for building validation logic.
    
    Reference: Gang of Four, Builder Pattern
    """
    
    def __init__(self, column_name: str):
        self.column_name = column_name
        self.chains: Dict[str, ValidationChain] = {}
        self.current_chain: Optional[ValidationChain] = None
    
    def chain(self, name: str) -> 'ValidationBuilder':
        """Start a new validation chain."""
        self.current_chain = ValidationChain(name)
        self.chains[name] = self.current_chain
        return self
    
    def stop_on_failure(self) -> 'ValidationBuilder':
        """Configure current chain to stop on first failure."""
        if self.current_chain:
            self.current_chain.set_stop_on_failure(True)
        return self
    
    # Number validators
    def is_numeric(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsNumericValidator("is_numeric", f"{self.column_name}.number")
        )
        return self
    
    def is_int(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsIntValidator("is_int", f"{self.column_name}.number.int")
        )
        return self
    
    def is_float(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsFloatValidator("is_float", f"{self.column_name}.number.float")
        )
        return self
    
    def in_range(self, min_val: float = None, max_val: float = None) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            NumericRangeValidator("in_range", f"{self.column_name}.number.range", 
                                min_val=min_val, max_val=max_val)
        )
        return self
    
    def decimal_format(self, decimal_places: int = 2) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            DecimalFormatValidator("decimal_format", f"{self.column_name}.number.float.decimal")
        )
        return self
    
    # String validators
    def is_string(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsStringValidator("is_string", f"{self.column_name}.string")
        )
        return self
    
    def length_range(self, min_length: int = None, max_length: int = None) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            StringLengthValidator("length_range", f"{self.column_name}.string.length")
        )
        return self
    
    def string_format(self, pattern: str) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            StringFormatValidator("string_format", f"{self.column_name}.string.format")
        )
        return self
    
    def in_value_list(self, allowed_values: List[Any]) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            ValueListValidator("in_value_list", f"{self.column_name}.string.values")
        )
        return self
    
    def contains_substring(self, substring: str, must_contain: bool = True) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            SubstringValidator("contains_substring", f"{self.column_name}.string.substring")
        )
        return self
    
    def is_uuid(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            UUIDValidator("is_uuid", f"{self.column_name}.string.uuid")
        )
        return self
    
    def inner_refer(self, reference_values: List[Any]) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            ReferenceValidator("inner_refer", f"{self.column_name}.string.reference", 
                             check_type="inner")
        )
        return self
    
    def outer_refer(self, reference_values: List[Any]) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            ReferenceValidator("outer_refer", f"{self.column_name}.string.reference",
                             check_type="outer")
        )
        return self
    
    # Boolean validators
    def is_boolean(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsBooleanValidator("is_boolean", f"{self.column_name}.boolean")
        )
        return self
    
    def boolean_mapping(self, true_values: List[Any] = None, 
                       false_values: List[Any] = None) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            BooleanMappingValidator("boolean_mapping", f"{self.column_name}.boolean.mapping")
        )
        return self
    
    # Date validators
    def is_date(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsDateValidator("is_date", f"{self.column_name}.date")
        )
        return self
    
    def date_format(self, date_format: str = "%Y-%m-%d") -> 'ValidationBuilder':
        self.current_chain.add_validator(
            DateFormatValidator("date_format", f"{self.column_name}.date.format")
        )
        return self
    
    def date_range(self, min_date: str = None, max_date: str = None) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            DateRangeValidator("date_range", f"{self.column_name}.date.range")
        )
        return self
    
    # Category validators
    def is_category(self) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            IsCategoryValidator("is_category", f"{self.column_name}.category")
        )
        return self
    
    # Custom validator
    def custom(self, name: str, validation_func: Callable) -> 'ValidationBuilder':
        self.current_chain.add_validator(
            CustomValidator(name, f"{self.column_name}.custom", validation_func)
        )
        return self
    
    def build(self) -> Dict[str, ValidationChain]:
        """Build and return all validation chains."""
        return self.chains


# ============= DATAFRAME VALIDATOR =============
class DataFrameValidator:
    """
    High-level validator for entire DataFrames.
    Orchestrates validation across multiple columns.
    """
    
    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.column_chains: Dict[str, Dict[str, ValidationChain]] = {}
    
    def add_column_validation(self, column: str, chains: Dict[str, ValidationChain]):
        """Register validation chains for a column."""
        if column not in self.df.columns:
            raise ValueError(f"Column '{column}' not found in DataFrame")
        self.column_chains[column] = chains
    
    def validate_column(self, column: str, chain_name: str = None) -> Dict[str, Any]:
        """Validate specific column with optional chain filter."""
        if column not in self.column_chains:
            raise ValueError(f"No validation defined for column '{column}'")
        
        data = self.df[column]
        results = {}
        
        chains = self.column_chains[column]
        if chain_name:
            if chain_name not in chains:
                raise ValueError(f"Chain '{chain_name}' not found for column '{column}'")
            chains = {chain_name: chains[chain_name]}
        
        for name, chain in chains.items():
            results[name] = chain.validate_and_summarize(data)
        
        return results
    
    def validate_all(self) -> Dict[str, Dict[str, Any]]:
        """Validate all registered columns."""
        all_results = {}
        
        for column in self.column_chains:
            all_results[column] = self.validate_column(column)
        
        return all_results
    
    def get_summary(self) -> pd.DataFrame:
        """Get summary of all validations as DataFrame."""
        summary_data = []
        
        results = self.validate_all()
        for column, chains in results.items():
            for chain_name, chain_result in chains.items():
                for result in chain_result['results']:
                    summary_data.append({
                        'column': column,
                        'chain': chain_name,
                        'validator': result.validator_name,
                        'type': result.validator_type,
                        'passed': result.passed,
                        'errors_count': len(result.invalid_indices)
                    })
        
        return pd.DataFrame(summary_data)


# ============= EXAMPLE USAGE =============
def example_usage():
    """Demonstrate the validation system."""
    
    # Sample data
    df = pd.DataFrame({
        'age': [25, 30, -5, 150, 45, 'invalid', None],
        'email': ['user@test.com', 'invalid-email', 'admin@test.com', 
                  'test@example.org', None, '', 'bad@'],
        'status': ['active', 'inactive', 'pending', 'invalid_status', 'active', None, 'active'],
        'score': [85.5, 92.3, 78.999, 88.12, 95.0, 101.5, 45.67],
        'is_premium': [True, False, 'yes', 'no', 1, 0, 'maybe']
    })
    
    print("=" * 80)
    print("DATAFRAME VALIDATION SYSTEM DEMO")
    print("=" * 80)
    print("\nOriginal DataFrame:")
    print(df)
    
    # Create validator
    validator = DataFrameValidator(df)
    
    # Build validation for 'age' column
    age_validation = (
        ValidationBuilder('age')
        .chain('type_check')
        .is_int()
        .chain('range_check')
        .in_range(min_val=0, max_val=120)
        .build()
    )
    validator.add_column_validation('age', age_validation)
    
    # Build validation for 'email' column
    email_validation = (
        ValidationBuilder('email')
        .chain('format_check')
        .is_string()
        .string_format(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')
        .build()
    )
    validator.add_column_validation('email', email_validation)
    
    # Build validation for 'status' column
    status_validation = (
        ValidationBuilder('status')
        .chain('value_check')
        .in_value_list(allowed_values=['active', 'inactive', 'pending'])
        .build()
    )
    validator.add_column_validation('status', status_validation)
    
    # Build validation for 'score' column
    score_validation = (
        ValidationBuilder('score')
        .chain('type_and_range')
        .is_float()
        .in_range(min_val=0, max_val=100)
        .decimal_format(decimal_places=2)
        .build()
    )
    validator.add_column_validation('score', score_validation)
    
    # Build validation for 'is_premium' column
    premium_validation = (
        ValidationBuilder('is_premium')
        .chain('boolean_check')
        .boolean_mapping()
        .build()
    )
    validator.add_column_validation('is_premium', premium_validation)
    
    # Run all validations
    print("\n" + "=" * 80)
    print("VALIDATION RESULTS")
    print("=" * 80)
    
    all_results = validator.validate_all()
    
    for column, chains in all_results.items():
        print(f"\n📋 Column: {column}")
        print("-" * 80)
        for chain_name, chain_result in chains.items():
            print(f"  Chain: {chain_name}")
            print(f"  Status: {'✓ ALL PASSED' if chain_result['all_passed'] else '✗ FAILURES DETECTED'}")
            print(f"  Validators: {chain_result['executed_validators']}/{chain_result['total_validators']}")
            
            if chain_result['failed_validators']:
                print(f"\n  Failed validations:")
                for result in chain_result['failed_validators']:
                    print(f"    • {result}")
                    print(f"      Invalid indices: {result.invalid_indices.tolist()}")
    
    # Get summary DataFrame
    print("\n" + "=" * 80)
    print("VALIDATION SUMMARY")
    print("=" * 80)
    summary = validator.get_summary()
    print(summary.to_string(index=False))
    
    # Example: Custom validator
    print("\n" + "=" * 80)
    print("CUSTOM VALIDATOR EXAMPLE")
    print("=" * 80)
    
    def custom_age_check(data: pd.Series, **kwargs) -> pd.Index:
        """Custom validation: age must be even number."""
        sr = data.dropna()
        numeric = pd.to_numeric(sr, errors='coerce')
        return numeric[numeric % 2 != 0].index
    
    custom_validation = (
        ValidationBuilder('age')
        .chain('custom_even_check')
        .custom('must_be_even', custom_age_check)
        .build()
    )
    
    # Validate with custom validator
    custom_chain = custom_validation['custom_even_check']
    custom_result = custom_chain.validate_and_summarize(df['age'])
    
    print(f"\nCustom validation (age must be even):")
    print(f"  Status: {'✓ PASSED' if custom_result['all_passed'] else '✗ FAILED'}")
    if custom_result['failed_validators']:
        for result in custom_result['failed_validators']:
            print(f"  {result}")
            print(f"  Odd numbers at indices: {result.invalid_indices.tolist()}")


if __name__ == "__main__":
    example_usage()